### Setup

In [34]:
# Library
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   
os.environ["CUDA_VISIBLE_DEVICES"]="0"
import torch
import random
import numpy as np
from datetime import datetime
import os
import json
import shutil
from datasets import Dataset, DatasetDict
from PIL import Image
from torchvision import transforms
from torchmetrics.image.fid import FrechetInceptionDistance

from huggingface_hub import login
from diffusers import DDPMScheduler, StableDiffusionPipeline, StableDiffusionPipeline
from torchmetrics.image.fid import FrechetInceptionDistance 
from metric import *

# GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device : ',device)   

# CONFIG
CURRENT_TIME = datetime.now().strftime("%Y%m%d_%H%M%S")
MASTER_SEED = 42

# TRAIN PARAMETER
IMG_SIZE = 512
BATCH_SIZE = 4
NUM_WORKERS = 4
EPOCHS = 50
LEARNING_RATE = 1e-04
TRAIN_DATA_SIZE = 1800
TEST_DATA_SIZE = 200

# PATH
CONFIG_PATH = '../config.json'
with open(CONFIG_PATH,'r') as f:
    config = json.load(f)
## DATA
TEST_INDEX = {'kfshion':0,'nike':1,'zara':2}
TEST_IMAGE_FOLDER = config.get("TEST_IMAGE_FOLDERS")[TEST_INDEX['kfshion']]
TEST_LABEL_FOLDER = config.get("TEST_LABEL_FOLDERS")[TEST_INDEX['kfshion']]
TEST_IMAGE_FILE = sorted([file for file in os.listdir(TEST_IMAGE_FOLDER) if file.endswith(('.jpg', '.jpeg', '.png'))], key=lambda x: int(x.split('.')[0]))
TEST_LABEL_FILE = sorted([file for file in os.listdir(TEST_LABEL_FOLDER) if file.endswith(('.json'))], key=lambda x: int(x.split('.')[0]))

## MODEL
PRE_TRAINED_MODEL_NAME="stablediffusionapi/deliberate-v2"

# HUGGINGFACE
HUGGING_FACE_TOKEN = config.get("HUGGING_FACE_TOKEN")
login(HUGGING_FACE_TOKEN)

device :  cuda


### Load Model

In [2]:
# load orginal model 
SAVE_WEIGHTS_PATH = '../Experiment/model_weights/FIGMA_weights_20250303_224850'
pipe = StableDiffusionPipeline.from_pretrained(PRE_TRAINED_MODEL_NAME, torch_dtype=torch.float16)
pipe.to("cuda")
# load fine-tunined model 
pipe.load_lora_weights(SAVE_WEIGHTS_PATH, safe_serialization=True) 
pipe.to("cuda")

tokenizer = pipe.tokenizer
text_encoder = pipe.text_encoder.to(torch.float16).to(device)  
vae = pipe.vae.to(torch.float16).to(device)  
unet = pipe.unet.to(torch.float16).to(device)  

noise_scheduler = DDPMScheduler.from_pretrained(
    PRE_TRAINED_MODEL_NAME,
    subfolder="scheduler"
)
if hasattr(noise_scheduler, "alphas_cumprod"):
    noise_scheduler.alphas_cumprod = noise_scheduler.alphas_cumprod.to(device)

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

### Preprocess Data

In [3]:
# Function to tokenize the text column in the given data sample and return token ID tensors.
def tokenize_captions(examples, caption_column='text', is_train=True):
    captions = []
    for caption in examples[caption_column]:
        # If there is only one caption
        if isinstance(caption, str):
            captions.append(caption)
        # If there are multiple captions, randomly select one
        elif isinstance(caption, (list, np.ndarray)):
            captions.append(random.choice(caption) if is_train else caption[0])  # Take a random caption if training, otherwise take the first one
        else:
            raise ValueError(
                f"Caption column `{caption_column}` should contain either strings or lists of strings."
            )
    
    inputs = tokenizer(
        captions, max_length=tokenizer.model_max_length, padding="max_length", truncation=True, return_tensors="pt"
    )

    return inputs.input_ids

# Transformation pipeline for preprocessing image data for training
transforms = transforms.Compose(
    [
        transforms.Resize(IMG_SIZE, interpolation=transforms.InterpolationMode.BILINEAR),
        transforms.CenterCrop(IMG_SIZE) if True else transforms.RandomCrop(IMG_SIZE),
        transforms.RandomHorizontalFlip() if True else transforms.Lambda(lambda x: x),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5]),  # Convert (0,1) -> (-1,1)
    ]
)

# Function to preprocess images by converting them to RGB, applying transformations, 
# and tokenizing the text column to construct a training dataset.
def preprocess_data(examples, image_column='image'):
    images = [image.convert("RGB") for image in examples[image_column]]
    # Image preprocessing
    examples["pixel_values"] = [transforms(image) for image in images]
    # Text preprocessing
    examples["input_ids"] = tokenize_captions(examples)
    return examples

# Function to batch images and token IDs, stacking them into PyTorch tensor format.
def collate_fn(examples):
    # (C, H, W), ..., (C, H, W) -> stack -> (N, C, H, W): Stack along N dimension
    pixel_values = torch.stack([example["pixel_values"] for example in examples])
    # Same as tensor.contiguous()
    pixel_values = pixel_values.to(memory_format=torch.contiguous_format).float()
    # {(77,), ..., (77,)}_N samples -> stack -> (N, 77)
    input_ids = torch.stack([example["input_ids"] for example in examples])

    return {"pixel_values": pixel_values, "input_ids": input_ids}


In [4]:
data = []
for image_file, label_file in zip(TEST_IMAGE_FILE, TEST_LABEL_FILE):
    image_path = os.path.join(TEST_IMAGE_FOLDER, image_file)
    label_path = os.path.join(TEST_LABEL_FOLDER, label_file)
    with open(image_path, 'rb') as image:
        image_data = Image.open(image)
        image_data = image_data.convert('RGB')
    with open(label_path, 'r') as label:
        label_data = json.load(label)
    data.append({
        'image':image_data,
        'text':label_data
    })
    
# Converting to Dataset
if TEST_IMAGE_FOLDER[25:] == "KFashion":
    dataset = Dataset.from_dict({
        'image': [item['image'] for item in data],
        'text': [item['text']['summary'] for item in data]
    })
elif TEST_IMAGE_FOLDER[25:] == "Nike" or TEST_IMAGE_FOLDER[25:] == "Zara":
    dataset = Dataset.from_dict({
        'image': [item['image'] for item in data],
        'text': [
            " | ".join([
                str(item['text']['Prompt']), 
                str(item['text']['Input']), 
                str(item['text']['Add_Info'])
            ])
            for item in data
        ]
    })

# Create Test DatasetDict and Apply Preprocessing
test_dataset = DatasetDict({'test': dataset})
test_dataset = test_dataset.with_transform(preprocess_data)
test_dataloader = torch.utils.data.DataLoader(
    test_dataset['test'],
    shuffle=False,
    collate_fn=collate_fn,
    batch_size=BATCH_SIZE
)
print(test_dataset)

DatasetDict({
    test: Dataset({
        features: ['image', 'text'],
        num_rows: 200
    })
})


### Test Model

In [ ]:
def calculate_fid(test_dataset, test_label_file, pipe):
    from torchvision import transforms as T
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    fid_metric = FrechetInceptionDistance(feature=64).to(device)
    fid_metric.reset()
    
    # Use transform to convert PIL images to uint8 tensors
    transform = T.PILToTensor()
    
    # Create a temporary folder (for storing created images)
    tmp_folder = '../Data/Total/tmp'
    os.makedirs(tmp_folder, exist_ok=True)
    for idx, filename in enumerate(test_label_file):
        prompt = test_dataset['test'][idx]['text'][16:]
        generated_image = pipe(prompt).images[0]
        # Save generate image   
        filename = filename.split('.')[0]
        generated_image.save(f'{tmp_folder}/{filename}.jpg')
    
        # real image
        real_image = test_dataset['test'][idx]['image']
        
        fake_tensor = transform(generated_image).unsqueeze(0).to(device)
        real_tensor = transform(real_image).unsqueeze(0).to(device)
        
        fid_metric.update(real_tensor, real=True)
        fid_metric.update(fake_tensor, real=False)
    
    fid_score = fid_metric.compute().item()
    return fid_score

fid_score = calculate_fid(test_dataset, TEST_LABEL_FILE, pipe)

In [6]:
# Compute precision and recall by comparing generated images with original images
GENERATE_IMAGE_FOLDER = '../Data/Total/tmp'
GENERATE_IMAGE_FILE = sorted([file for file in os.listdir(GENERATE_IMAGE_FOLDER)], key=lambda x: int(x.split('.')[0]))

# Initialize TensorFlow session and evaluator
config = tf.ConfigProto(allow_soft_placement=True)
config.gpu_options.allow_growth = True
sess = tf.Session(config=config)
evaluator = Evaluator(sess)
evaluator.warmup()

# Prepare batch generator (original images)
seed_batches = batch_generator(TEST_IMAGE_FILE, TEST_IMAGE_FOLDER, batch_size=64)
# Prepare batch generator (generated images)
augment_batches = batch_generator(GENERATE_IMAGE_FILE, GENERATE_IMAGE_FOLDER, batch_size=64)

# Extract activations for original images (pool_3 features)
seed_activations = evaluator.compute_activations(seed_batches) 
augment_activations = evaluator.compute_activations(augment_batches)

# Compute precision and recall
precision, recall = evaluator.compute_prec_recall(seed_activations[0], augment_activations[0])

# Remove folder if it exists
shutil.rmtree(GENERATE_IMAGE_FOLDER)

2025-03-10 12:47:45.529371: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:995] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-03-10 12:47:45.553766: W tensorflow/core/common_runtime/gpu/gpu_device.cc:1960] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...
2025-03-10 12:47:45.904375: W tensorflow/core/framework/op_def_util.cc:369] Op BatchNormWithGlobalNormalization is deprecated. It will cease to work in GraphDef version 9. Use tf.nn.batch_normalization().


  0%|          | 0/1 [00:00<?, ?it/s]

2025-03-10 12:47:46.492745: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:375] MLIR V1 optimization pass is not enabled


0it [00:00, ?it/s]

0it [00:00, ?it/s]